# 응용 — CONTEXUS 꾸민 공부 기록 판별기 설계하기

4장의 2층 신경망으로 **"이 공부 기록은 진짜인가, 꾸민 것인가"**를 맞히는 모델을 처음부터 끝까지 설계해 본다.
데이터는 `make_dummy.py`가 만든 **가짜 기록**이다. 실제 사용자 기록은 쓰지 않는다.

설계 순서 = 진행표 ①~⑤

| Part | 진행표 | 묻는 것 |
|---|---|---|
| 1 | ① 문제 정의 | 모델이 **무엇을** 맞히나 |
| 2 | ② 입력 | 모델에 **무엇을** 보여 주나 |
| 3 | ③ 정답 | 맞혔는지 **어떻게** 아나 |
| 4 | ④ 모델 | 4장 `TwoLayerNet`으로 학습 |
| 5 | ⑤ 평가 | 얼마나 틀려도 되나 |
| 6 | ⭐ 앱으로 | 어떤 기록이 중요했나 → 앱이 남겨야 할 기록 |

**표시**: 🔁 배치·행렬 연습 / ⭐ 책 범위 밖 / ⚠️ 함정 / 🔮 실행 전에 예측하기 / 💭 정답 없는 생각 칸
`# 여기에 작성`을 채우고 확인 셀에서 `통과!`가 나오면 성공. 막히면 `design_solutions.ipynb`.

## 0. 준비
**이 셀부터** 실행하자. 4장 응용(Fashion-MNIST)과 같은 준비 셀이다.

In [ ]:
import sys, os
sys.path.append(os.pardir)
import time
import numpy as np
import matplotlib.pyplot as plt
from common.functions import sigmoid, softmax, cross_entropy_error, sigmoid_grad
from common.gradient import numerical_gradient
from make_dummy import load_dummy, FEATURES, FEATURE_NAMES, LABEL_NAMES

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def check(got, want, name='답'):
    if got is None:
        raise AssertionError(f'{name}: 아직 안 채웠어요 (None)')
    if isinstance(want, (tuple, str, bool)):
        ok = got == want
    else:
        ok = np.shape(got) == np.shape(want) and np.allclose(got, want)
    mark = '✅' if ok else '❌'
    print(f'{name} = {got!r}   (정답 {want!r}) {mark}')
    assert ok, f'{name} 이(가) 틀렸어요'

class TwoLayerNet:   # 4장 5-2
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        a1 = np.dot(x, self.params['W1']) + self.params['b1']
        z1 = sigmoid(a1)
        a2 = np.dot(z1, self.params['W2']) + self.params['b2']
        return softmax(a2)

    def loss(self, x, t):
        return cross_entropy_error(self.predict(x), t)

    def accuracy(self, x, t):
        y = np.argmax(self.predict(x), axis=1)
        t = np.argmax(t, axis=1)
        return np.sum(y == t) / float(x.shape[0])

    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        grads = {}
        for key in ('W1', 'b1', 'W2', 'b2'):
            grads[key] = numerical_gradient(loss_W, self.params[key])
        return grads

    def gradient(self, x, t):   # ⭐ 5장 오차역전파법 — 실행만
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
        batch_num = x.shape[0]
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = softmax(a2)
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)
        dz1 = sigmoid_grad(a1) * np.dot(dy, W2.T)
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)
        return grads

print('준비 완료')

# Part 1. ① 문제 정의 — 모델이 무엇을 맞히나
답의 모양은 세 가지가 있었다: **둘 중 하나** / **확률** / **점수 여러 개**.
이번에는 **확률**로 간다. 4장 MNIST가 숫자 10개 각각의 확률을 냈듯이, 이번에는 `[진짜일 확률, 꾸몄을 확률]` 두 칸을 낸다. 마지막 층은 똑같이 **소프트맥스**.

## 1-1. 출력 칸 정하기
- `output_size`: 신경망 출력 칸 수 (숫자)
- `fake_index`: 그중 '꾸밈'은 몇 번 칸? `LABEL_NAMES = ['진짜', '꾸밈']`을 보고 (숫자)

In [ ]:
output_size = None   # 여기에 작성
fake_index = None    # 여기에 작성

In [ ]:
# 확인
check(output_size, 2, 'output_size')
check(fake_index, LABEL_NAMES.index('꾸밈'), 'fake_index')
print('통과!')

## 1-2. 🔮 출력 읽기
어떤 기록에 대해 신경망이 `y = [0.3, 0.7]`을 냈다.
- `prob_fake`: 꾸몄을 확률은? (숫자)
- `prob_sum`: 두 칸을 더하면? 소프트맥스의 성질 (숫자)

In [ ]:
prob_fake = None   # 여기에 작성
prob_sum = None    # 여기에 작성

In [ ]:
# 확인
y = np.array([0.3, 0.7])
check(prob_fake, y[fake_index], 'prob_fake')
check(prob_sum, y.sum(), 'prob_sum')
print('통과!')

# Part 2. ② 입력 — 모델에 무엇을 보여 주나
기록 한 묶음(한 사람이 한 문서를 공부한 기록 전체)을 **숫자 8개**로 줄여서 넣는다. MNIST가 사진 한 장을 숫자 784개로 넣은 것과 같다.

## 2-1. 🔮 불러오기 전에 모양 예측하기
`load_dummy()`는 훈련 기록 3000개, 시험 기록 1000개를 준다. 정답은 원-핫.
- `shape_x`: `x_train.shape` (튜플)
- `shape_t`: `t_train.shape` (튜플)

In [ ]:
shape_x = None   # 여기에 작성
shape_t = None   # 여기에 작성

In [ ]:
# 불러오기 (실행만) + 확인
(x_train, t_train), (x_test, t_test) = load_dummy(one_hot_label=True)
print(x_train.shape, t_train.shape, x_test.shape, t_test.shape)
check(shape_x, x_train.shape, 'shape_x')
check(shape_t, t_train.shape, 'shape_t')
print('통과!')

## 2-2. 입력 8칸 살펴보기 (실행만)
앱이 실제로 기록할 수 있는 값들이다. `page_match`만은 **질문 글과 쪽 글을 읽어야** 구할 수 있다 (⭐ 책 범위 밖: 임베딩). 여기서는 더미가 이미 숫자로 준다.

In [ ]:
for j, (name, desc, unit) in enumerate(FEATURES):
    print(f'{j}  {name:15s} {desc} ({unit})')
print()
print('첫 기록:', np.round(x_train[0], 2), '→ 정답', LABEL_NAMES[np.argmax(t_train[0])])

## 2-3. 🔁 진짜 · 꾸밈별 평균 — 한 줄씩
원-핫 정답을 숫자로 바꾸고, **불리언 마스크**로 진짜 기록만 / 꾸밈 기록만 골라 열마다 평균을 낸다.
- `t_label`: `t_train`을 정답 숫자로 (`argmax`, `axis` 주의) → 모양 (3000,)
- `mean_honest`: 진짜 기록들의 열별 평균 → 모양 (8,)
- `mean_fake`: 꾸민 기록들의 열별 평균 → 모양 (8,)

In [ ]:
t_label = None       # 여기에 작성
mean_honest = None   # 여기에 작성
mean_fake = None     # 여기에 작성

In [ ]:
# 확인 + 비교표
check(t_label.shape, (3000,), 't_label.shape')
check(mean_honest.shape, (8,), 'mean_honest.shape')
check(round(float(mean_honest[0]), 2), 0.14, 'mean_honest[0] (붙여넣기 비율)')
check(round(float(mean_fake[0]), 2), 0.56, 'mean_fake[0] (붙여넣기 비율)')
print()
for j, name in enumerate(FEATURE_NAMES):
    print(f'{name:15s} 진짜 {mean_honest[j]:8.2f} | 꾸밈 {mean_fake[j]:8.2f}')
print('통과!')

## 2-4. 🔮 혼자서는 쓸모없는 입력
위 표에서 **진짜와 꾸밈의 평균이 거의 같은** 입력이 하나 있다. 꾸밈 B(띄워 두기형)가 날짜를 채우기 때문이다.
- `weak_feature`: 그 입력의 이름 (문자열, `FEATURE_NAMES` 중 하나)

In [ ]:
weak_feature = None   # 여기에 작성

In [ ]:
# 확인
check(weak_feature, 'study_days', 'weak_feature')
print('통과! "공부한 날 수"만 보면 꾸밈을 못 잡는다 → 다른 기록과 같이 봐야 한다')

## 2-5. 그림으로 보기 (실행만)
겹치는 부분이 클수록 그 입력 하나로는 구분이 어렵다.

In [ ]:
plt.figure(figsize=(12, 3.2))
for k, name in enumerate(['paste_ratio', 'typing_speed', 'interval_cv', 'study_days']):
    j = FEATURE_NAMES.index(name)
    plt.subplot(1, 4, k + 1)
    plt.hist(x_train[t_label == 0, j], bins=30, alpha=0.6, density=True, label='진짜')
    plt.hist(x_train[t_label == 1, j], bins=30, alpha=0.6, density=True, label='꾸밈')
    plt.title(name)
    if k == 0:
        plt.legend()
plt.tight_layout()
plt.show()

# Part 3. ③ 정답 — 맞혔는지 어떻게 아나
더미에서는 정답을 우리가 정해 뒀다. **실제 앱에서는 이게 가장 어렵다.**

## 3-1. 꾸밈 비율
- `fake_rate`: 훈련 기록 중 꾸민 기록의 비율 (원-핫 `t_train`의 '꾸밈' 칸 평균)

In [ ]:
fake_rate = None   # 여기에 작성

In [ ]:
# 확인
check(fake_rate, 0.2, 'fake_rate')
print('통과!')

## 3-2. ⚠️ 아무것도 안 하는 모델의 정확도
"모든 기록은 진짜다"라고만 답하는 모델이 있다. 학습도 안 했다.
- `baseline_acc`: 이 모델의 정확도는? (숫자 — `fake_rate`로 계산)

이 숫자보다 높아야 비로소 모델이 **무언가를 배운 것**이다.

In [ ]:
baseline_acc = None   # 여기에 작성

In [ ]:
# 확인
check(baseline_acc, 0.8, 'baseline_acc')
print('통과! 정확도 80%는 아무것도 안 해도 나온다')

## 3-3. 💭 실제 앱에서는 정답을 어디서 얻을까
떠오르는 대로 적어 보자. (채점 없음)

참고 후보: ① 팀이 직접 꾸며 본 기록 (공격자 역할) ② 전문가가 기록을 보고 매긴 판정 ③ 나중의 실제 실력(시험 · 면접)과 맞춰 보기

In [ ]:
내_생각 = ''   # 여기에 작성

# Part 4. ④ 모델 — 4장 TwoLayerNet으로 학습
## 4-1. ⚠️ 입력 크기 맞추기 (정규화)
MNIST는 `normalize=True`로 0~1에 맞췄다. 이번 입력은 단위가 제각각이다.
- 🔮 `biggest`: 2-3 표에서 값이 **가장 큰** 입력의 이름 (문자열)

큰 값이 섞이면 그 입력이 가중치 갱신을 휘두른다. 그래서 열마다 **평균 0, 표준편차 1**로 맞춘다 (표준화).
- `mean`, `std`: **훈련** 데이터의 열별 평균 · 표준편차 → 모양 (8,)
- `x_train_n`, `x_test_n`: 표준화한 값 — ⚠️ 시험 데이터도 **훈련의** `mean`, `std`로 바꾼다 (시험 데이터는 '미래'라서 미리 볼 수 없다)

In [ ]:
biggest = None     # 여기에 작성

mean = None        # 여기에 작성
std = None         # 여기에 작성
x_train_n = None   # 여기에 작성
x_test_n = None    # 여기에 작성

In [ ]:
# 확인
check(biggest, 'dwell_sec', 'biggest')
check(np.round(x_train_n.mean(axis=0), 6), np.zeros(8), '훈련 열별 평균')
check(np.round(x_train_n.std(axis=0), 6), np.ones(8), '훈련 열별 표준편차')
check(bool(np.allclose(x_test_n, (x_test - x_train.mean(axis=0)) / x_train.std(axis=0))), True, 'x_test_n을 훈련 기준으로 바꿨나')
print('통과!')

## 4-2. 🔮 매개변수 개수
`TwoLayerNet(input_size=8, hidden_size=10, output_size=2)`의 매개변수(가중치 + 편향)는 모두 몇 개?
- `n_params` (숫자) — W1, b1, W2, b2 크기를 더한다

In [ ]:
n_params = None   # 여기에 작성

In [ ]:
# 확인
net_tmp = TwoLayerNet(input_size=8, hidden_size=10, output_size=2)
check(n_params, sum(v.size for v in net_tmp.params.values()), 'n_params')
print('통과! MNIST 신경망(784→50→10)은 39,760개였다. 이번엔 아주 작아서 수치 미분으로도 학습된다')

## 4-3. 학습 — 수치 미분으로 (4장 그대로)
4장 5-3의 학습 반복을 채우자. 이번에는 ⭐ `gradient` 대신 **4장에서 배운 `numerical_gradient`**를 쓴다. 30~60초 걸린다.
1. 미니배치: `batch_mask`, `x_batch`, `t_batch`
2. 기울기: `grad`
3. 갱신: 키 4개 반복

In [ ]:
np.random.seed(0)
network = TwoLayerNet(input_size=8, hidden_size=10, output_size=2)

iters_num = 2000
train_size = x_train_n.shape[0]
batch_size = 100
learning_rate = 1.0

train_loss_list = []
train_acc_list = []
test_acc_list = []

start = time.time()
for i in range(iters_num):
    # 1. 미니배치
    batch_mask = None   # 여기에 작성
    x_batch = None      # 여기에 작성
    t_batch = None      # 여기에 작성

    # 2. 기울기 (수치 미분)
    grad = None         # 여기에 작성

    # 3. 갱신 (키 4개)
    # 여기에 작성

    train_loss_list.append(network.loss(x_batch, t_batch))

    # 4. 200번마다 정확도
    if i % 200 == 0:
        train_acc_list.append(network.accuracy(x_train_n, t_train))
        test_acc_list.append(network.accuracy(x_test_n, t_test))
        print(f'{i:5d}  train acc {train_acc_list[-1]:.4f} | test acc {test_acc_list[-1]:.4f}')

print(f'걸린 시간 {time.time() - start:.1f}초')

In [ ]:
# 확인
check(len(train_loss_list), 2000, '손실 기록 수')
check(bool(train_loss_list[-1] < train_loss_list[0]), True, '손실이 줄었나')
check(bool(test_acc_list[-1] > baseline_acc), True, '아무것도 안 하는 모델보다 나은가')
print('통과!')

## 4-4. 학습 곡선 (실행만)
처음 몇백 번 동안 정확도가 0.8에 머무는 구간이 보이면, 그때는 "전부 진짜"라고만 답하던 시기다 (3-2).

In [ ]:
plt.figure(figsize=(11, 3.5))
plt.subplot(1, 2, 1)
plt.plot(train_loss_list)
plt.xlabel('반복'); plt.ylabel('손실'); plt.title('훈련 손실')
plt.subplot(1, 2, 2)
xs = np.arange(len(train_acc_list)) * 200
plt.plot(xs, train_acc_list, label='train')
plt.plot(xs, test_acc_list, label='test', linestyle='--')
plt.axhline(baseline_acc, color='gray', linewidth=0.8, label='아무것도 안 함 (0.8)')
plt.xlabel('반복'); plt.ylabel('정확도'); plt.ylim(0.7, 1.0); plt.legend(); plt.title('정확도')
plt.tight_layout()
plt.show()

## 4-5. ⭐ 5장 미리보기 — 오차역전파로 바꾸면 (실행만)
같은 학습을 `gradient`로 돌린다. 결과는 거의 같고, 시간만 크게 줄어든다. 실제 앱 데이터가 커지면 5장이 필요한 이유다.

In [ ]:
np.random.seed(0)
net_fast = TwoLayerNet(input_size=8, hidden_size=10, output_size=2)
start = time.time()
for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    grad = net_fast.gradient(x_train_n[batch_mask], t_train[batch_mask])
    for key in ('W1', 'b1', 'W2', 'b2'):
        net_fast.params[key] -= learning_rate * grad[key]
print(f'오차역전파: {time.time() - start:.2f}초 | test acc {net_fast.accuracy(x_test_n, t_test):.4f}')

# Part 5. ⑤ 평가 — 얼마나 틀려도 되나
## 5-1. 시험 정확도
- `test_acc`: `network`의 시험 데이터 정확도

In [ ]:
test_acc = None   # 여기에 작성

In [ ]:
# 확인
check(test_acc, network.accuracy(x_test_n, t_test), 'test_acc')
print('통과!')

## 5-2. 🔁 두 종류의 실수
정확도 하나로는 **어떤 실수**를 했는지 모른다. 채용에 쓰는 기록이라 두 실수의 무게가 다르다.

| | 모델: 진짜 | 모델: 꾸밈 |
|---|---|---|
| **실제: 진짜** | 맞음 | ❗ **오탐** — 성실한 사람을 꾸몄다고 함 |
| **실제: 꾸밈** | 놓침 — 꾸민 사람을 통과시킴 | 맞음 |

- `y_pred`: 시험 기록마다 모델이 고른 칸 (`predict` → `argmax`) → 모양 (1000,)
- `t_true`: 실제 정답 숫자 → 모양 (1000,)
- `false_alarm_rate`: **진짜 기록 중** 꾸밈으로 잘못 잡은 비율
- `miss_rate`: **꾸민 기록 중** 진짜로 통과시킨 비율

힌트: `np.sum((t_true == 0) & (y_pred == 1)) / np.sum(t_true == 0)`

In [ ]:
y_pred = None             # 여기에 작성
t_true = None             # 여기에 작성
false_alarm_rate = None   # 여기에 작성
miss_rate = None          # 여기에 작성

In [ ]:
# 확인
yp = np.argmax(network.predict(x_test_n), axis=1); tt = np.argmax(t_test, axis=1)
check(false_alarm_rate, np.mean(yp[tt == 0] == 1), 'false_alarm_rate')
check(miss_rate, np.mean(yp[tt == 1] == 0), 'miss_rate')
print(f'오탐 {false_alarm_rate:.3f} | 놓침 {miss_rate:.3f}')
print('통과!')

## 5-3. 판정 문턱 바꾸기
지금은 꾸밈 확률이 0.5를 넘으면 꾸밈이라고 한다. 이 **문턱**을 올리면 "아주 확실할 때만" 꾸밈이라고 한다.
- 🔮 `predict_fa`: 문턱을 0.5 → 0.9로 올리면 오탐률은? (`'늘어난다'` / `'줄어든다'`)
- `rates(th)`: 문턱 `th`에서 `(오탐률, 놓침률)`을 돌려주는 함수 — 5-2와 같은데, `y_pred` 대신 `prob_fake > th`를 쓴다

In [ ]:
predict_fa = None   # 여기에 작성

prob = network.predict(x_test_n)[:, fake_index]   # 시험 기록마다 꾸몄을 확률

def rates(th):
    flagged = None   # 여기에 작성 (꾸밈이라고 판정했나: True/False 배열)
    fa = None        # 여기에 작성
    miss = None      # 여기에 작성
    return fa, miss

In [ ]:
# 확인 + 표
check(predict_fa, '줄어든다', 'predict_fa')
check(rates(0.5), (false_alarm_rate, miss_rate), 'rates(0.5) = 5-2 결과')
print()
print(' 문턱 |  오탐률 |  놓침률')
for th in (0.5, 0.7, 0.9, 0.99):
    fa, miss = rates(th)
    print(f' {th:4.2f} | {fa:7.3f} | {miss:7.3f}')
print('통과! 오탐을 줄이면 놓침이 늘어난다 — 둘을 동시에 0으로 만들 수는 없다')

## 5-4. 💭 문턱은 누가 정하나
채용에 쓰이는 기록이라면 어느 실수를 더 줄여야 할까? 문턱을 몇으로 둘지, 왜 그런지 적어 보자. (채점 없음)

In [ ]:
내_문턱 = None   # 여기에 작성 (숫자)
이유 = ''        # 여기에 작성

# Part 6. ⭐ 앱으로 — 어떤 기록이 중요했나
모델이 무엇을 보고 판단했는지 알아보는 쉬운 방법: **입력 한 칸만 마구 섞어서** 정확도가 얼마나 떨어지는지 본다. 많이 떨어질수록 그 칸이 중요했다. (순열 중요도)

## 6-1. 한 칸씩 섞어 보기
- 반복문 안 `x_shuf`: `x_test_n`을 **복사**한 뒤 `j`번 열만 `np.random.permutation`으로 섞는다
- `drop`: 원래 정확도 `base_acc` − 섞은 뒤 정확도

⚠️ `.copy()` 없이 `x_shuf = x_test_n`이라고 쓰면 원본이 망가진다 (같은 배열을 가리킴).

In [ ]:
np.random.seed(0)
base_acc = network.accuracy(x_test_n, t_test)
drops = []
for j in range(8):
    x_shuf = None   # 여기에 작성 (복사)
    # 여기에 작성 (j번 열만 섞기)
    drop = None     # 여기에 작성
    drops.append(drop)
drops = np.array(drops)

In [ ]:
# 확인 + 그림
check(drops.shape, (8,), 'drops.shape')
check(network.accuracy(x_test_n, t_test), base_acc, '원본이 그대로인가')
order = np.argsort(drops)
plt.figure(figsize=(7, 3.5))
plt.barh([FEATURE_NAMES[j] for j in order], drops[order])
plt.xlabel('섞었을 때 떨어진 정확도'); plt.title('어떤 기록이 판정에 중요했나')
plt.tight_layout(); plt.show()
for j in order[::-1]:
    print(f'{FEATURE_NAMES[j]:15s} {drops[j]:+.3f}')
print('통과!')

## 6-2. 정리 — 이 연습에서 앱으로 가져갈 것
1. **앱이 지금부터 기록해 둘 값 목록** = 입력 8칸. 실제 데이터가 쌓여야 진짜 모델을 만들 수 있다.
2. **날 수(`study_days`) 하나만으로는 꾸밈을 못 잡는다.** 여러 기록이 맞물릴 때 강해진다.
3. **정확도보다 오탐률.** 문턱은 모델이 아니라 **사람(회사)이 정하는 결정**이다.

**🔴 한계** — 이 모델은 `make_dummy.py`에 우리가 적은 규칙을 다시 찾아냈을 뿐이다. 실제 꾸미기를 잡는 능력이 아니다.
실제 앱 모델로 가려면: ① 진짜 정답 데이터 (Part 3) ② 질문 글 이해 (`page_match` — 임베딩) ③ 5장 오차역전파 (4-5)